# 2 — Build the phase-normalised dataset

Turns every trial into per-step kinematics on a common phase base: 0-100% of
ground contact followed by 0-100% of flight. Athletes whose contact and flight
times differ are then compared at the same point of the same phase, instead of
at the same fraction of a stride.

Contact and flight *durations* are kept as separate scalar features — dividing
them out is what makes the shapes comparable, but the durations themselves are
determinants of sprint speed, not nuisance.

Equivalent to `python -m sprint build`.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sprint import config as C, events, features, figures, io, model, skeleton

In [ ]:
from sprint.cli import build
build(None)

## What came out

In [ ]:
for phase in ("accel", "topspeed"):
    df = pd.read_csv(C.DATA_DIR / f"features_{phase}.csv")
    curves = np.load(C.DATA_DIR / f"curves_{phase}.npy")
    print(f"{phase}: {len(df)} athletes, {df.shape[1]} columns, curves {curves.shape}")
pd.read_csv(C.DATA_DIR / "features_topspeed.csv").describe().T

## Phase base sanity

At 60 Hz a top-speed contact is only ~6-7 raw frames, which is why the base is 20+20 points and not 101 — a denser grid would manufacture resolution the capture rate does not hold. Phases shorter than 3 raw frames are NaN rather than interpolated.

In [ ]:
for phase in ("accel", "topspeed"):
    curves = np.load(C.DATA_DIR / f"curves_{phase}.npy")
    frac = np.isnan(curves).any(axis=2).mean(axis=0)
    print(f"{phase}: NaN fraction  contact {frac[:C.N_CONTACT].mean():.3f}  "
          f"flight {frac[C.N_CONTACT:].mean():.3f}")